# **Deeper Player Scouting**

This notebook investigates the reasons behind player performance rather than relying on total points alone. It examines form, expected involvement, efficiency, minutes, ownership, and upcoming fixtures to create a more informed scouting shortlist.

## **1. Data Setup**

This notebook continues the work from `Fantasy_Scout.ipynb`. It independently retrieves the same player, team, and fixture data so the deeper scouting analysis can be run as a separate workflow without depending on the previous notebook's active kernel.


In [ ]:
import pandas as pd
import requests

BASE_URL = "https://fantasy.premierleague.com/api"
MIN_MINUTES = 90


def fetch_json(url: str, timeout: int = 20):
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()


bootstrap = fetch_json(f"{BASE_URL}/bootstrap-static/")
fixtures_payload = fetch_json(f"{BASE_URL}/fixtures/")

team_map = {
    team["id"]: team["name"]
    for team in bootstrap.get("teams", [])
}
position_map = {
    position["id"]: position.get("singular_name_short", position.get("singular_name"))
    for position in bootstrap.get("element_types", [])
}

players_df = pd.DataFrame(bootstrap.get("elements", []))
players_df = players_df.rename(columns={
    "id": "player_id",
    "web_name": "name",
    "element_type": "position_id",
    "now_cost": "price_tenths",
    "selected_by_percent": "ownership_pct",
})

players_df["team"] = players_df["team"].map(team_map)
players_df["position"] = players_df["position_id"].map(position_map)
players_df["price"] = pd.to_numeric(players_df["price_tenths"], errors="coerce") / 10

numeric_columns = [
    "total_points",
    "form",
    "expected_goals",
    "expected_assists",
    "ict_index",
    "minutes",
    "goals_scored",
    "assists",
    "clean_sheets",
    "saves",
    "goals_conceded",
    "bonus",
    "ownership_pct",
]
for column in numeric_columns:
    players_df[column] = pd.to_numeric(players_df[column], errors="coerce").fillna(0)

players_df = players_df.rename(columns={
    "expected_goals": "xg",
    "expected_assists": "xa",
})

current_event = int(bootstrap.get("current_event", 1))
upcoming_fixtures = [
    fixture for fixture in fixtures_payload
    if fixture.get("event") is not None
    and fixture.get("finished") is False
    and current_event <= int(fixture["event"]) <= current_event + 4
]

fixture_rows = []
for fixture in upcoming_fixtures:
    fixture_rows.extend([
        {
            "event": fixture["event"],
            "team_id": fixture["team_h"],
            "team_name": team_map.get(fixture["team_h"]),
            "opponent": team_map.get(fixture["team_a"]),
            "is_home": True,
            "difficulty": fixture.get("team_h_difficulty"),
        },
        {
            "event": fixture["event"],
            "team_id": fixture["team_a"],
            "team_name": team_map.get(fixture["team_a"]),
            "opponent": team_map.get(fixture["team_h"]),
            "is_home": False,
            "difficulty": fixture.get("team_a_difficulty"),
        },
    ])

fixtures_df = pd.DataFrame(fixture_rows)
print(f"Loaded {len(players_df)} players and {len(fixtures_df)} team-fixture rows.")
print(f"Early-season minimum minutes filter: {MIN_MINUTES}")


Loaded 651 players and 60 team-fixture rows.
Early-season minimum minutes filter: 90


## **2. Recent Form and Reliability**

This section identifies players combining points, form, and dependable playing time.

In [14]:
# **2.1 Form and minutes**

form_view = players_df[
    (players_df["minutes"] >= MIN_MINUTES)
    & (players_df["status"] == "a")
].copy()

form_view["points_per_90"] = form_view["total_points"] / (form_view["minutes"] / 90).replace(0, pd.NA)

form_view = form_view.sort_values(
    ["form", "points_per_90", "minutes"],
    ascending=False,
)

print(form_view[[
    "name", "position", "team", "price", "form", "minutes",
    "total_points", "points_per_90", "ownership_pct",
]].head(20))


             name position       team  price  form  minutes  total_points  \
506   B.Fernandes      MID    Man Utd   12.0  12.5      180            25   
472        Cherki      MID   Man City    7.7  11.0      108            22   
317         Ajayi      DEF  Hull City    4.1  10.0      153            20   
11           Saka      MID    Arsenal    9.5  10.0      157            20   
7       Calafiori      DEF    Arsenal    5.6  10.0      170            20   
170        Palmer      MID    Chelsea    9.6  10.0      172            20   
178    João Pedro      FWD    Chelsea    7.7  10.0      180            20   
341      Tzolakis      GKP  Hull City    4.6  10.0      180            20   
117     M.Sangaré      MID  Brentford    5.7   9.0      165            18   
9           White      DEF    Arsenal    5.5   9.0      180            18   
263     Tarkowski      DEF    Everton    6.0   9.0      180            18   
534        Elanga      MID  Newcastle    6.1   8.5      156            17   

## **3. Expected Performance and Efficiency**

Expected goals and expected assists help show whether a player's underlying attacking involvement supports their actual returns.

In [15]:
# **3.1 Actual returns compared with expected involvement**

expected_view = players_df[
    (players_df["minutes"] >= MIN_MINUTES)
    & (players_df["position"].isin(["MID", "FWD"]))
].copy()

expected_view["goal_difference"] = expected_view["goals_scored"] - expected_view["xg"]
expected_view["assist_difference"] = expected_view["assists"] - expected_view["xa"]
expected_view["attacking_involvement"] = expected_view["xg"] + expected_view["xa"]

print(expected_view.sort_values("attacking_involvement", ascending=False)[[
    "name", "position", "team", "price", "goals_scored", "xg",
    "assists", "xa", "goal_difference", "assist_difference",
]].head(20))


              name position            team  price  goals_scored    xg  \
506    B.Fernandes      MID         Man Utd   12.0             3  2.10   
507         Mbeumo      MID         Man Utd    8.0             1  2.26   
155         Rogers      MID         Chelsea    7.5             1  1.10   
449           Isak      FWD       Liverpool    9.0             1  2.08   
280          Barry      FWD         Everton    5.5             1  1.93   
178     João Pedro      FWD         Chelsea    7.7             2  1.86   
114         Thiago      FWD       Brentford    8.0             0  1.80   
471          Foden      MID        Man City    7.0             0  0.99   
247        Nketiah      FWD  Crystal Palace    5.5             0  0.96   
438     Szoboszlai      MID       Liverpool    7.0             1  1.01   
481        Haaland      FWD        Man City   15.5             2  1.40   
472         Cherki      MID        Man City    7.7             2  0.35   
11            Saka      MID         Ar

In [16]:
# **3.2 Points and attacking output per 90 minutes**

per90_view = players_df[players_df["minutes"] >= MIN_MINUTES].copy()
minutes_factor = per90_view["minutes"] / 90
per90_view["points_per_90"] = per90_view["total_points"] / minutes_factor.replace(0, pd.NA)
per90_view["xg_per_90"] = per90_view["xg"] / minutes_factor.replace(0, pd.NA)
per90_view["xa_per_90"] = per90_view["xa"] / minutes_factor.replace(0, pd.NA)
per90_view["points_per_price"] = per90_view["total_points"] / per90_view["price"].replace(0, pd.NA)

print(per90_view.sort_values("points_per_90", ascending=False)[[
    "name", "position", "team", "price", "minutes", "points_per_90",
    "xg_per_90", "xa_per_90", "points_per_price",
]].head(20))


            name position           team  price  minutes  points_per_90  \
472       Cherki      MID       Man City    7.7      108      18.333333   
506  B.Fernandes      MID        Man Utd   12.0      180      12.500000   
345        Mendy      DEF      Hull City    4.0      121      11.900826   
317        Ajayi      DEF      Hull City    4.1      153      11.764706   
11          Saka      MID        Arsenal    9.5      157      11.464968   
7      Calafiori      DEF        Arsenal    5.6      170      10.588235   
170       Palmer      MID        Chelsea    9.6      172      10.465116   
178   João Pedro      FWD        Chelsea    7.7      180      10.000000   
341     Tzolakis      GKP      Hull City    4.6      180      10.000000   
117    M.Sangaré      MID      Brentford    5.7      165       9.818182   
534       Elanga      MID      Newcastle    6.1      156       9.807692   
437        Gakpo      MID      Liverpool    7.0      160       9.562500   
123    De Cuyper      DEF

## **4. Fixture Suitability**

This section connects player teams with the next five gameweeks to identify favourable schedules.

In [17]:
# **4.1 Team fixture difficulty**

fixture_summary = (
    fixtures_df.groupby("team_name", as_index=False)
    .agg(
        fixtures=("event", "count"),
        average_difficulty=("difficulty", "mean"),
        home_fixtures=("is_home", "sum"),
    )
    .sort_values(["average_difficulty", "fixtures"])
)

print(fixture_summary.head(20))

         team_name  fixtures  average_difficulty  home_fixtures
13       Liverpool         3            2.333333              1
1      Aston Villa         3            2.666667              1
4         Brighton         3            2.666667              2
7   Crystal Palace         3            2.666667              1
12           Leeds         3            2.666667              2
14        Man City         3            2.666667              2
16       Newcastle         3            2.666667              2
3        Brentford         3            3.000000              2
8          Everton         3            3.000000              2
17   Nott'm Forest         3            3.000000              2
18           Spurs         3            3.000000              2
0          Arsenal         3            3.333333              1
2      Bournemouth         3            3.333333              2
5          Chelsea         3            3.333333              1
6    Coventry City         3            

## **5. Scouting Shortlist**

The shortlist combines playing time, form, output, value, and ownership. It is a starting point for human judgement rather than an automatic transfer recommendation.

In [18]:
# **5.1 Balanced shortlist**

scouting_view = players_df[
    (players_df["minutes"] >= MIN_MINUTES)
    & (players_df["status"] == "a")
].copy()

scouting_view["points_per_price"] = scouting_view["total_points"] / scouting_view["price"].replace(0, pd.NA)
scouting_view["points_per_90"] = scouting_view["total_points"] / (scouting_view["minutes"] / 90).replace(0, pd.NA)
scouting_view["attacking_involvement"] = scouting_view["xg"] + scouting_view["xa"]

scouting_view["scouting_score"] = (
    scouting_view["points_per_90"].rank(pct=True)
    + scouting_view["points_per_price"].rank(pct=True)
    + scouting_view["form"].rank(pct=True)
    + scouting_view["attacking_involvement"].rank(pct=True)
)

shortlist = scouting_view.sort_values("scouting_score", ascending=False)
print(shortlist[[
    "name", "position", "team", "price", "form", "minutes",
    "total_points", "points_per_90", "points_per_price",
    "attacking_involvement", "ownership_pct", "scouting_score",
]].head(20))


             name position           team  price  form  minutes  total_points  \
472        Cherki      MID       Man City    7.7  11.0      108            22   
123     De Cuyper      DEF       Brighton    4.7   8.5      167            17   
506   B.Fernandes      MID        Man Utd   12.0  12.5      180            25   
178    João Pedro      FWD        Chelsea    7.7  10.0      180            20   
7       Calafiori      DEF        Arsenal    5.6  10.0      170            20   
11           Saka      MID        Arsenal    9.5  10.0      157            20   
170        Palmer      MID        Chelsea    9.6  10.0      172            20   
317         Ajayi      DEF      Hull City    4.1  10.0      153            20   
437         Gakpo      MID      Liverpool    7.0   8.5      160            17   
97   Lewis-Potter      MID      Brentford    5.5   8.0      177            16   
117     M.Sangaré      MID      Brentford    5.7   9.0      165            18   
99         Kayode      DEF  

In [ ]:
### **6.4 Attacker assessment**

attackers = players_df[
    (players_df["position"] == "FWD")
    & (players_df["minutes"] >= MIN_MINUTES)
].copy()

attackers["points_per_90"] = attackers["total_points"] / (attackers["minutes"] / 90)
attackers["goal_threat"] = attackers["xg"] + attackers["goals_scored"]
attackers["returns"] = attackers["goals_scored"] + attackers["assists"]

attackers = attackers.sort_values(
    ["goal_threat", "returns", "points_per_90"],
    ascending=False,
)

print(attackers[[
    "name", "team", "price", "minutes", "total_points", "form",
    "goals_scored", "assists", "xg", "goal_threat", "returns",
    "points_per_90", "ownership_pct",
]].head(20))


In [ ]:
### **6.3 Midfielder assessment**

midfielders = players_df[
    (players_df["position"] == "MID")
    & (players_df["minutes"] >= MIN_MINUTES)
].copy()

midfielders["points_per_90"] = midfielders["total_points"] / (midfielders["minutes"] / 90)
midfielders["attacking_involvement"] = midfielders["xg"] + midfielders["xa"]
midfielders["returns"] = midfielders["goals_scored"] + midfielders["assists"]

midfielders = midfielders.sort_values(
    ["attacking_involvement", "returns", "points_per_90"],
    ascending=False,
)

print(midfielders[[
    "name", "team", "price", "minutes", "total_points", "form",
    "goals_scored", "assists", "xg", "xa", "attacking_involvement",
    "points_per_90", "ownership_pct",
]].head(20))


In [ ]:
### **6.2 Defender assessment**

defenders = players_df[
    (players_df["position"] == "DEF")
    & (players_df["minutes"] >= MIN_MINUTES)
].copy()

defenders["points_per_90"] = defenders["total_points"] / (defenders["minutes"] / 90)

# Defensive reliability and attacking contribution are both relevant for defenders.
defenders["defender_output"] = (
    defenders["clean_sheets"]
    + defenders["goals_scored"]
    + defenders["assists"]
    + defenders["bonus"]
)

defenders = defenders.sort_values(
    ["defender_output", "points_per_90", "minutes"],
    ascending=False,
)

print(defenders[[
    "name", "team", "price", "minutes", "total_points",
    "clean_sheets", "goals_scored", "assists", "bonus",
    "defender_output", "points_per_90", "ownership_pct",
]].head(20))


In [ ]:
### **6.1 Goalkeeper assessment**

keepers = players_df[
    (players_df["position"] == "GKP")
    & (players_df["minutes"] >= MIN_MINUTES)
].copy()

keepers["points_per_90"] = keepers["total_points"] / (keepers["minutes"] / 90)
keepers["saves_per_90"] = keepers["saves"] / (keepers["minutes"] / 90)

keepers = keepers.sort_values(
    ["clean_sheets", "saves_per_90", "points_per_90"],
    ascending=False,
)

print(keepers[[
    "name", "team", "price", "minutes", "total_points",
    "clean_sheets", "saves", "saves_per_90", "goals_conceded",
    "points_per_90", "ownership_pct",
]].head(15))


## **6. Position-Based Player Assessments**

Each position is assessed using the statistics most relevant to its FPL scoring role. These are screening views for comparison, not automatic transfer recommendations.


## **7. Interpretation**

Use the outputs to distinguish reliable starters from short-term performers. A strong scouting candidate should ideally combine regular minutes, sustainable underlying numbers, acceptable price, and a favourable fixture run. Ownership and recent form add context, but neither should be used in isolation.
